# People Map Algorithm Demo

End-to-end walkthrough of the SixDegrees People Map pipeline: data fetch → interaction scoring → combined distance matrix → t-SNE projection → origin translation → visualization.

> **Launch from project root:** `cd /path/to/sixDegrees && jupyter notebook`
> All cells must be run in order.

In [ ]:
import os
import sys
import math

# Resolve backend relative to project root
# Assumption: Jupyter launched from project root (sixDegrees/)
BACKEND_DIR = os.path.abspath("backend")
if not os.path.isdir(BACKEND_DIR):
    raise RuntimeError(f"backend/ not found at {BACKEND_DIR}. Launch Jupyter from the project root.")

sys.path.insert(0, BACKEND_DIR)

from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join(BACKEND_DIR, ".env"))

if not os.getenv("SUPABASE_URL") or not os.getenv("SUPABASE_KEY"):
    raise EnvironmentError("SUPABASE_URL and SUPABASE_KEY must be set in backend/.env")

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

print(f"Backend path: {BACKEND_DIR}")
print(f"Supabase URL configured: {bool(os.getenv('SUPABASE_URL'))}")

## Stage 1: Data Fetch

Reads all user profiles and pairwise interaction counts from Supabase.

In [ ]:
from services.map_pipeline.data_fetcher import fetch_all
from config.supabase import get_supabase_client

profiles, interactions = fetch_all()

# Build a display_name lookup from Supabase (UserProfile model omits display_name)
sb = get_supabase_client()
name_rows = sb.table("user_profiles").select("user_id,display_name").execute().data
names_lookup = {r["user_id"]: r["display_name"] for r in name_rows}

print(f"Users loaded: {len(profiles)}")
print(f"Interaction pairs: {len(interactions)}")
print("\nSample profiles:")
for p in profiles[:3]:
    print(f"  {names_lookup.get(p.id, p.id[:8])} — interests={p.interests[:2]}, city={p.city}")
print("\nSample interactions:")
sample_pairs = list(interactions.items())[:3]
for (uid_a, uid_b), counts in sample_pairs:
    print(f"  {names_lookup.get(uid_a, uid_a[:8])}/{names_lookup.get(uid_b, uid_b[:8])}: "
          f"likes={counts.get('likes', 0)}, comments={counts.get('comments', 0)}, dms={counts.get('dms', 0)}")

## Stage 2: Interaction Scoring

Converts raw interaction counts to a normalized score matrix in [0, 1] using min-max normalization with 95th-percentile clipping. Weights come from `config/algorithm.py` INTERACTION_WEIGHTS.

In [ ]:
from services.map_pipeline.interaction import compute_interaction_scores
from config.algorithm import INTERACTION_WEIGHTS

user_ids = [p.id for p in profiles]
interaction_matrix = compute_interaction_scores(user_ids, interactions)

print(f"Interaction score matrix shape: {interaction_matrix.shape}")
print(f"Score range: [{interaction_matrix.min():.4f}, {interaction_matrix.max():.4f}]")
print(f"Mean score: {interaction_matrix.mean():.4f}")
print(f"INTERACTION_WEIGHTS: {INTERACTION_WEIGHTS}")

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(interaction_matrix, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_title("Interaction Score Matrix\n(brighter = more interaction)")
ax.set_xlabel("User index")
ax.set_ylabel("User index")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Stage 3: Combined Distance Matrix

Combines profile distance (α=0.6) and interaction score (β=0.4): `distance = α × profile_dist + β × (1 - interaction_score)`. Result is a symmetric NxN matrix with values in [0, 1].

In [ ]:
from services.map_pipeline.scoring import build_combined_distance_matrix
from config.algorithm import ALPHA, BETA

dist_matrix = build_combined_distance_matrix(profiles, interaction_matrix)

print(f"Distance matrix shape: {dist_matrix.shape}")
print(f"Symmetric: {np.allclose(dist_matrix, dist_matrix.T)}")
print(f"Diagonal zeros: {np.allclose(np.diag(dist_matrix), 0)}")
print(f"Range: [{dist_matrix.min():.4f}, {dist_matrix.max():.4f}]")
print(f"α={ALPHA}, β={BETA}")

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(dist_matrix, cmap="viridis", vmin=0, vmax=1)
ax.set_title(f"Combined Distance Matrix (α={ALPHA}, β={BETA})\n(darker = more similar / more interaction)")
ax.set_xlabel("User index")
ax.set_ylabel("User index")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Stage 4: t-SNE Projection

t-SNE reduces the NxN precomputed distance matrix to 2D coordinates. `metric='precomputed'` tells sklearn the input is already distances. Perplexity is computed dynamically as `min(30, max(5, sqrt(N)))`.

In [ ]:
from services.map_pipeline.tsne_projector import project_tsne

raw_coords = project_tsne(dist_matrix)

print(f"t-SNE output shape: {raw_coords.shape}")
print(f"x range: [{raw_coords[:,0].min():.2f}, {raw_coords[:,0].max():.2f}]")
print(f"y range: [{raw_coords[:,1].min():.2f}, {raw_coords[:,1].max():.2f}]")

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(raw_coords[:, 0], raw_coords[:, 1], alpha=0.7, s=60)
for i, p in enumerate(profiles):
    ax.annotate(names_lookup.get(p.id, p.id[:8]),
                (raw_coords[i, 0], raw_coords[i, 1]),
                textcoords="offset points", xytext=(5, 5), fontsize=7)
ax.set_title("t-SNE Raw Coordinates (before origin translation)")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
plt.tight_layout()
plt.show()

## Stage 5: Origin Translation + Tier Assignment

Shifts all coordinates so the requesting user (Alex Rivera) is at (0, 0). Assigns tiers: Tier 1 = 5 nearest, Tier 2 = next 10, Tier 3 = remainder.

In [ ]:
from services.map_pipeline.origin_translator import translate_and_assign_tiers

DEMO_CENTER_USER = "3561ceb0-d433-437d-8a4f-08da002dff50"  # Alex Rivera

results = translate_and_assign_tiers(user_ids, raw_coords, DEMO_CENTER_USER)

# results is a list of dicts: {user_id, x, y, tier}
center = next((r for r in results if r["user_id"] == DEMO_CENTER_USER), None)
print(f"Center user (Alex Rivera) at: ({center['x']:.6f}, {center['y']:.6f})")
print(f"Total coordinate rows: {len(results)}")
print(f"Tier 1: {sum(1 for r in results if r['tier'] == 1)}, "
      f"Tier 2: {sum(1 for r in results if r['tier'] == 2)}, "
      f"Tier 3: {sum(1 for r in results if r['tier'] == 3)}")

TIER_COLORS = {1: "#e74c3c", 2: "#3498db", 3: "#95a5a6"}
TIER_NAMES  = {1: "Tier 1 (closest)", 2: "Tier 2", 3: "Tier 3"}

fig, ax = plt.subplots(figsize=(11, 9))
for tier in [1, 2, 3]:
    pts = [r for r in results if r["tier"] == tier]
    if pts:
        ax.scatter([p["x"] for p in pts], [p["y"] for p in pts],
                   c=TIER_COLORS[tier], label=TIER_NAMES[tier], s=80, alpha=0.8)
    for p in pts:
        ax.annotate(names_lookup.get(p["user_id"], "?"),
                    (p["x"], p["y"]),
                    textcoords="offset points", xytext=(5, 5), fontsize=7)
ax.scatter(0, 0, c="black", marker="*", s=200, label="Alex Rivera (you)", zorder=5)
ax.axhline(0, color="gray", linewidth=0.5, linestyle="--", alpha=0.4)
ax.axvline(0, color="gray", linewidth=0.5, linestyle="--", alpha=0.4)
ax.set_title("Alex Rivera's People Map (after origin translation + tier assignment)")
ax.legend(fontsize=9)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
plt.tight_layout()
plt.show()

## Sensitivity Demonstration

Increasing the interaction count between two users makes them appear closer on the map. We bump Alex Rivera ↔ Skyler Thompson likes_count from the seed value to 60, re-run the full pipeline, and compare.

In [ ]:
from services.map_pipeline import run_pipeline_for_user

BUMP_USER_A = "3561ceb0-d433-437d-8a4f-08da002dff50"  # Alex Rivera
BUMP_USER_B = "af17902c-723d-4a32-a5a1-93d9fb7777ee"  # Skyler Thompson
BUMP_LIKES  = 60

def get_current_coords(center_id):
    """Fetch current map_coordinates from DB after pipeline run."""
    sb = get_supabase_client()
    rows = (
        sb.table("map_coordinates")
        .select("other_user_id, x, y, tier")
        .eq("center_user_id", center_id)
        .eq("is_current", True)
        .execute()
    ).data
    return [
        {"user_id": r["other_user_id"], "x": r["x"], "y": r["y"],
         "tier": r["tier"], "display_name": names_lookup.get(r["other_user_id"], "?")}
        for r in rows
    ]

def dist_between(coords, uid_a, uid_b):
    lookup = {c["user_id"]: c for c in coords}
    if uid_a not in lookup or uid_b not in lookup:
        return None
    a, b = lookup[uid_a], lookup[uid_b]
    return math.sqrt((a["x"] - b["x"])**2 + (a["y"] - b["y"])**2)

def set_likes(uid_a, uid_b, count):
    uid_a, uid_b = min(uid_a, uid_b), max(uid_a, uid_b)
    get_supabase_client().table("interactions").upsert(
        {"user_id_a": uid_a, "user_id_b": uid_b, "likes_count": count},
        on_conflict="user_id_a,user_id_b",
    ).execute()

def get_likes(uid_a, uid_b):
    uid_a, uid_b = min(uid_a, uid_b), max(uid_a, uid_b)
    sb = get_supabase_client()
    rows = (sb.table("interactions").select("likes_count")
            .eq("user_id_a", uid_a).eq("user_id_b", uid_b).execute()).data
    return rows[0]["likes_count"] if rows else 0

# Baseline
print("Running baseline pipeline...")
run_pipeline_for_user(DEMO_CENTER_USER)
baseline_coords = get_current_coords(DEMO_CENTER_USER)
baseline_dist = dist_between(baseline_coords, BUMP_USER_A, BUMP_USER_B)
original_likes = get_likes(BUMP_USER_A, BUMP_USER_B)
print(f"Baseline distance(Alex, Skyler) = {baseline_dist:.4f}")
print(f"Current likes_count = {original_likes}")

# Bump
print(f"\nBumping likes_count to {BUMP_LIKES}...")
set_likes(BUMP_USER_A, BUMP_USER_B, BUMP_LIKES)

try:
    print("Re-running pipeline with bumped interaction...")
    run_pipeline_for_user(DEMO_CENTER_USER)
    bumped_coords = get_current_coords(DEMO_CENTER_USER)
    bumped_dist = dist_between(bumped_coords, BUMP_USER_A, BUMP_USER_B)
    print(f"Bumped  distance(Alex, Skyler) = {bumped_dist:.4f}")
    print(f"\nChange: {bumped_dist - baseline_dist:+.4f} ({'CLOSER' if bumped_dist < baseline_dist else 'farther'})")

    # Side-by-side plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    for ax, coords, title in [
        (ax1, baseline_coords, f"Baseline (likes={original_likes})"),
        (ax2, bumped_coords,   f"After bump (likes={BUMP_LIKES})"),
    ]:
        for tier in [1, 2, 3]:
            pts = [c for c in coords if c["tier"] == tier]
            if pts:
                ax.scatter([p["x"] for p in pts], [p["y"] for p in pts],
                           c=TIER_COLORS[tier], label=TIER_NAMES[tier], s=80, alpha=0.8)
            for p in pts:
                ax.annotate(p["display_name"], (p["x"], p["y"]),
                            textcoords="offset points", xytext=(5, 5), fontsize=7)
        ax.scatter(0, 0, c="black", marker="*", s=200, zorder=5)
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.set_xlabel("t-SNE dim 1")
    plt.suptitle("Sensitivity: Alex\u2013Skyler interaction count bumped", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
finally:
    print(f"\nRestoring likes_count to {original_likes}...")
    set_likes(BUMP_USER_A, BUMP_USER_B, original_likes)
    print("Done. Supabase counts restored.")